In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

In [63]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df[['Serangan_Jantung']]

In [64]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [65]:
params= {
'C': np.random.uniform(0.001,100.0,size=100),
'penalty': ['l1','l2','elasticnet'],
'l1_ratio': np.random.uniform(0.0, 1.0, size=100)
}
log_reg = LogisticRegression(max_iter=10000,solver='saga')
random_search = RandomizedSearchCV(estimator=log_reg,param_distributions=params,
                                   n_iter=20,cv=5,scoring='accuracy',
                                   random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train.values.ravel())

print(f'Hyperparameter Terbaik: {random_search.best_params_}')
print(f'Akurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')

best_model = random_search.best_estimator_
tes_accuracy = best_model.score(X_test, y_test)

y_pred_test= best_model.predict(X_test)
y_prob_test= best_model.predict_proba(X_test)

print(f'Akurasi Data Test: {tes_accuracy*100:.2f}%')

Hyperparameter Terbaik: {'penalty': 'elasticnet', 'l1_ratio': np.float64(0.19416509054217768), 'C': np.float64(92.74471930702758)}
Akurasi Validasi Terbaik: 75.00%
Akurasi Data Test: 74.67%


In [66]:
def evaluate_model(y_pred,y_test,y_prob,evaluate,model_name='Logistic Regression'):
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)
    roc_auc = roc_auc_score(y_test,y_prob[:,1])
    logloss = log_loss(y_test,y_prob)
    
    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [67]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    acc_train = df_eval_train['Accuracy'].values[0]
    acc_test = df_eval_test['Accuracy'].values[0]
    gap = acc_train - acc_test

    if acc_train < 0.60 and acc_test < 0.60:
        status = 'Underfitting (akurasi rendah)'
    elif gap > 0.05 :
        status = f'Overfitting (gap:{gap:.2f%})'
    elif gap < -0.05:
        status = 'Overfitting / Data leakage (Test > Train)'
    else:
        status = 'Good fit'
    df_combined['Status'] = status
    return df_combined

In [68]:
df_eval_train = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate= 'Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate= 'Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)
df_eval

,Model,Evaluated,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Log Loss,Status
0,Logistic Regression,Training,0.746667,0.785311,0.785311,0.785311,0.836112,0.492262,Good fit
1,Logistic Regression,Test,0.746667,0.785311,0.785311,0.785311,0.836112,0.492262,Good fit


## Melakukan Prediksi Dengan Data Random

In [69]:
from sklearn.preprocessing import StandardScaler
data_5_pasien = {
    'Usia': [63, 45, 56, 38, 50],
    'Jenis_Kelamin': [3, 1, 0, 1, 0],
    'Tipe_Nyeri_Dada': [-0.3, 1.2, 0.08, -1.05, -0.48],
    'Tekanan_Darah_Rest': [140, 130, 122, 110, 120],
    'Kolesterol': [69, 100, 122, 115, 200],
    'Gula_Darah_Puasa': [1, 0, 0, 0, 1],
    'EKG_Rest': [1, 1, 1, 1, 1],
    'Detak_Jantung_Max': [-0.98, 0.77, -1.05, 0.29, -1.58],
    'Angina_Olahraga': [44, 60, 56, 75, 14],
    'Oldpeak_ST': [19, 55, 73, 76, 18],
    'Kemiringan_ST': [1.28, 1.0, 1.5, 1.5, 2.0],
    'Jumlah_Pembuluh_Darah': [46, 7, 4, 0, 3],
    'Thalassemia': [81, 7, 1, 4, 1],
    'BMI': [100.77, 31.24, 24.5, 28.1, 29.3]
}
feauter_numerik = ['Usia','Tekanan_Darah_Rest','Kolesterol','Detak_Jantung_Max','Oldpeak_ST','BMI']
scaler = StandardScaler()
data_pasien = pd.DataFrame(data_5_pasien)
data_pasien[feauter_numerik] = scaler.fit_transform(data_pasien[feauter_numerik])
data_pasien

,Usia,Jenis_Kelamin,Tipe_Nyeri_Dada,Tekanan_Darah_Rest,Kolesterol,Gula_Darah_Puasa,EKG_Rest,Detak_Jantung_Max,Angina_Olahraga,Oldpeak_ST,Kemiringan_ST,Jumlah_Pembuluh_Darah,Thalassemia,BMI
0,1.458427,3,-0.30,1.548888,-1.202446,1,1,-0.529744,44,-1.154448,1.28,46,81,1.994286
1,-0.625040,1,1.20,0.556011,-0.488350,0,1,1.442708,60,0.268844,1.00,7,7,-0.396945
2,0.648190,0,0.08,-0.238290,0.018428,0,1,-0.608642,56,0.980490,1.50,4,1,-0.628743
3,-1.435277,1,-1.05,-1.429743,-0.142819,0,1,0.901693,75,1.099098,1.50,0,4,-0.504934
4,-0.046299,0,-0.48,-0.436866,1.815186,1,1,-1.206014,14,-1.193984,2.00,3,1,-0.463664


In [ ]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
# data_pasien_baru = df_x.sample(2, random_state=10)
data_pasien_baru = data_pasien
prediksi_pasien = best_model.predict(data_pasien_baru)
probabilitas_pasien = best_model.predict_proba(data_pasien_baru)

hasil_df = data_pasien_baru.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

hasil_df[['Peluang Aman(%)','Peluang Terkena (%)','Prediksi Serangan Jantung']]

=== Melakukan Prediksi Data Pasien Baru ===


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,0.000000,100.000000,1
1,0.000000,100.000000,1
2,0.000000,100.000000,1
3,0.000000,100.000000,1
4,0.000003,99.999997,1
